# 05 多项式根拓展：综合除法、Bairstow 与逐次压缩

多项式求根是非线性方程求根的重要特例。与一般函数不同，多项式有明确的代数结构：Horner 格式能稳定地计算函数值和导数；找到一个根后，可用综合除法把次数降低；若希望保持实系数，还可以寻找二次因子，一次分离一对实根或共轭复根。

本节不追求替代成熟的 `numpy.roots`，而是展示多项式求根方法背后的数值结构和误差来源。


In [ ]:
import pathlib
import sys

import numpy as np

ROOT = pathlib.Path.cwd()
while ROOT.name != "py-sc" and ROOT != ROOT.parent:
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from py_sc import (
    bairstow_quadratic_factor,
    newton_polynomial_roots,
    polynomial_value_and_derivative,
    synthetic_division,
)


## Horner 求值和导数

对降幂系数

$$
p(x)=a_0x^n+a_1x^{n-1}+\cdots+a_n,
$$

Horner 格式把求值写成嵌套乘加。若同时维护导数递推，就能在一次扫描中得到 $p(x)$ 和 $p'(x)$，这是 Newton 多项式求根的基础。


In [ ]:
def teaching_horner_value_derivative(coefficients, x):
    coefficients = np.asarray(coefficients, dtype=complex)
    value = coefficients[0]
    derivative = 0.0j
    for coefficient in coefficients[1:]:
        derivative = derivative * x + value
        value = value * x + coefficient
    return value, derivative

coefficients = [1.0, -6.0, 11.0, -6.0]
value, derivative = teaching_horner_value_derivative(coefficients, 2.0)
official_value, official_derivative = polynomial_value_and_derivative(coefficients, 2.0)

print("teaching value, derivative:", value, derivative)
print("official value, derivative:", official_value, official_derivative)


## 综合除法与逐次压缩

若 $r$ 是 $p(x)$ 的一个根，则

$$
p(x)=(x-r)q(x).
$$

综合除法可以在线性时间内得到 $q(x)$ 和余数。逐次压缩法的思路是：先用 Newton 法在当前多项式上找一个根，再除去该线性因子，然后继续处理低一阶多项式。


In [ ]:
quotient, remainder = synthetic_division(coefficients, 1.0)
roots_result = newton_polynomial_roots(coefficients, initial_guesses=[0.8, 2.2, 3.2], tolerance=1e-12)

print("quotient after root 1:", np.array2string(quotient.real, precision=8), "remainder:", f"{abs(remainder):.3e}")
print("roots by Newton deflation:", np.array2string(np.sort(roots_result.roots.real), precision=12))
print("residual norm:", f"{roots_result.residual_norm:.3e}")


## Bairstow 型二次因子迭代

对实系数多项式，复根通常成共轭对出现。若直接寻找线性复根，压缩后的系数会进入复数。Bairstow 思想改为寻找实二次因子

$$
x^2+ux+v,
$$

让多项式除以该因子的余式趋于零。经典 Bairstow 法可写成专门的合成除法递推；这里用同一思想构造余式关于 $(u,v)$ 的 Newton 迭代，便于把目标看成“让线性余式两个系数同时为零”。


In [ ]:
poly_with_quadratic = np.array([1.0, -3.0, 3.0, -3.0, 2.0])  # (x^2 - 3x + 2)(x^2 + 1)
factor_result = bairstow_quadratic_factor(
    poly_with_quadratic,
    linear_coefficient=-2.5,
    constant_coefficient=1.5,
    tolerance=1e-12,
)

print("quadratic factor:", np.array2string(np.asarray(factor_result.factor, dtype=float), precision=12))
print("factor roots:", np.array2string(np.sort(np.asarray(factor_result.roots, dtype=float)), precision=12))
print("iterations:", factor_result.iterations, "residual:", f"{factor_result.residual_norm:.3e}")


## 数值风险

多项式求根的额外风险主要来自压缩和系数敏感性：

* 根的顺序会影响后续压缩多项式的系数误差；
* 重根附近 Newton 法会变慢，且导数可能接近零；
* 高次多项式的系数扰动可能导致根显著移动；
* Bairstow 型方法需要求解二元 Newton 修正，Jacobian 可能病态或奇异；
* 共轭复根、实根和近重根混在一起时，需要更成熟的全局策略。

因此，教学实现适合说明方法结构；实际生产计算更应优先使用经过充分验证的多项式特征值或伴随矩阵算法。


In [ ]:
wilkinson_like = np.poly(np.arange(1.0, 6.0))
roots = newton_polynomial_roots(wilkinson_like, initial_guesses=np.arange(0.8, 5.8), tolerance=1e-10)
print("degree-5 integer roots:", np.array2string(np.sort(roots.roots.real), precision=8))
print("residual norm:", f"{roots.residual_norm:.3e}")


## 本节小结

多项式求根可以利用比一般函数更多的结构。Horner 格式给出高效的函数值和导数计算；综合除法允许每找到一个根就降低次数；Newton 逐次压缩适合展示“求一个根、降一次阶”的基本流程；Bairstow 型二次因子迭代则适合保持实系数并一次提取一对根。它们都对初值、重根和舍入误差敏感，因此本章把它们作为结构性拓展，而不是全局最稳健的高次多项式求根器。
